# الفصل 20: قوانين التوسّع: Chinchilla

> طبقة الحدس تخيّل أن لديك ميزانية حوسبة ثابتة (لنقل: ما يعادل 1000 وحدة معالجة رسومية (GPU) لمدة شهر).

- **الجزء:** 5 — بناء LLM من الصفر
- **الفصل:** 20
- **المستوى:** متقدم
- **زمن القراءة:** 25 دقيقة
- **المتطلبات:** الفصل 19: تدريب Tokenizer مخصص
- **الوسوم:** Chinchilla، Scaling Laws، Compute-Optimal، Tokens

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch20-scaling-laws.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, scipy
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("scipy", "scipy")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-05-building-llms%2Fch20-scaling-laws.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-05-building-llms/ch20-scaling-laws.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> طبقة الحدس تخيّل أن لديك ميزانية حوسبة ثابتة (لنقل: ما يعادل 1000 وحدة معالجة رسومية (GPU) لمدة شهر).

> **بيانات الفصل**
>
> - **الجزء:** 5 — بناء LLM من الصفر
> - **المستوى:** متقدم
> - **زمن القراءة:** 25 دقيقة
> - **المتطلبات:** الفصل 19: تدريب Tokenizer مخصص


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم صياغة المشكلة رياضيًا
- فهم قيد ميزانية الحوسبة
- فهم كود: تقدير الخسارة ونقطة التحسين الأمثل


## 1. لماذا يهمّك هذا الموضوع؟

> 🟢 **طبقة الحدس**

تخيّل أن لديك ميزانية حوسبة ثابتة (لنقل: ما يعادل 1000 وحدة معالجة رسومية (GPU) لمدة شهر). السؤال الذي يواجه كل فريق قبل الشروع في تدريب LLM:

> هل أبني نموذجًا **كبيرًا** (عدد معاملات ضخم) وأدرّبه على بيانات **قليلة نسبيًا**؟ أم أبني نموذجًا **أصغر** وأدرّبه على بيانات **أكثر بكثير**؟

هذا بالضبط السؤال الذي أجابت عنه ورقة **Chinchilla** الشهيرة من DeepMind (Hoffmann et al., 2022) بعنوان *"Training Compute-Optimal Large Language Models"*، والتي صححت افتراضًا سائدًا سابقًا من ورقة Kaplan et al. (2020) التي اعتُمِدت في تدريب GPT-3.

**الاكتشاف الصادم**: نموذج GPT-3 (175 مليار معامل) كان في الواقع **مُدرَّبًا تدريبًا ناقصًا (Undertrained)** نسبة لحجمه! دُرِّب على ~300 مليار رمز فقط، بينما الحجم الأمثل وفق قوانين Chinchilla كان يستدعي تدريبه على عدد أكبر بكثير من الرموز، أو بدلاً من ذلك، بناء نموذج أصغر بكثير بنفس ميزانية الحوسبة وتدريبه على بيانات كافية ليصل لأداء مماثل أو أفضل.

فعليًا، أظهرت تجربة Chinchilla (70 مليار معامل، مُدرَّب على 1.4 تريليون رمز) أداءً **يتفوق** على Gopher (280 مليار معامل، مُدرَّب على 300 مليار رمز فقط) رغم أن Chinchilla أصغر بـ 4 أضعاف! وذلك لأن ميزانية الحوسبة المستخدمة في تدريب كليهما كانت متقاربة، لكن Chinchilla وزّعتها بشكل "أمثل" بين حجم النموذج وكمية البيانات.


## 2. الخريطة الذهنية


```text
Chinchilla Scaling Laws
│
├── المتغيرات الأساسية
│     N: عدد معاملات النموذج
│     D: عدد رموز التدريب (Tokens)
│     C: ميزانية الحوسبة (FLOPs) ≈ 6ND
│     L: دالة الخسارة النهائية (Loss)
│
├── العلاقة الإمبريقية
│     L(N, D) = E + A/N^α + B/D^β
│     (قابلة للنمذجة بتجارب متعددة الأحجام)
│
├── النتيجة الأشهر (القاعدة العملية)
│     D ≈ 20 × N  (لكل معامل، نحتاج ~20 رمز تدريب)
│
├── التخصيص الأمثل لميزانية حوسبة C
│     N_opt ∝ C^0.5,   D_opt ∝ C^0.5
│     (كلاهما ينمو بنفس المعدل تقريبًا مع زيادة الميزانية)
│
└── تحذير مهم: هذا "الأمثل للتدريب" ≠ "الأمثل للاستنتاج"
      (نماذج مثل LLaMA تتجاوز D=20N عمدًا لتحسين كفاءة الاستدلال لاحقًا)
```


## 3. الشرح المتدرج

### 3.1 صياغة المشكلة رياضيًا

> 🟡 **طبقة الآلية**

نفترض أن خسارة التدريب النهائية لنموذج لغوي (Cross-Entropy Loss) تتبع قانون قوة (Power Law) في كل من عدد المعاملات N وعدد رموز التدريب D:

$$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}$$

حيث:
- $E$: ثابت يمثل "إنتروبيا اللغة الطبيعية" نفسها (الحد الأدنى النظري للخسارة الذي لا يمكن تجاوزه مهما كبر النموذج أو كثرت البيانات، لأن اللغة البشرية نفسها تحتوي غموضًا/عشوائية متأصلة)
- $A, \alpha$: ثوابت تصف كيف تتناقص الخسارة مع زيادة حجم النموذج N (بافتراض بيانات كافية)
- $B, \beta$: ثوابت تصف كيف تتناقص الخسارة مع زيادة حجم بيانات التدريب D (بافتراض نموذج كافي الحجم)

هذه الثوابت (E, A, α, B, β) تُقدَّر تجريبيًا عبر تدريب **مئات النماذج** بأحجام N وD مختلفة على نطاق واسع جدًا (من ملايين المعاملات إلى مليارات)، ثم ملاءمة (Fitting) هذه المعادلة على النتائج الفعلية المرصودة.

### 3.2 قيد ميزانية الحوسبة

ميزانية الحوسبة الكلية اللازمة لتدريب نموذج بحجم N معامل على D رمز تُقدَّر تقريبيًا بالصيغة الشهيرة:

$$C \approx 6 \times N \times D \quad \text{(FLOPs)}$$

> 🔴 **طبقة التعمّق**: من أين يأتي المُعامل 6 تحديدًا؟ لكل رمز تدريب واحد، يحتاج النموذج لإجراء **تمريرة أمامية (Forward Pass)** و**تمريرة خلفية (Backward Pass)**. التمريرة الأمامية تتطلب تقريبًا $2N$ عملية FLOP لكل رمز (كل معامل يُضرَب ويُجمَع مرة تقريبًا)، والتمريرة الخلفية تتطلب تقريبًا ضعف ذلك ($4N$) لأنها تحسب التدرجات بالنسبة للأوزان والمدخلات معًا. المجموع: $2N + 4N = 6N$ عملية لكل رمز واحد، وبضربها في D رمز إجمالي نحصل على $C \approx 6ND$.

الآن السؤال المحوري لدالة Chinchilla: **بالنظر إلى ميزانية حوسبة ثابتة C، كيف نوزّع N وD لتدنية L(N,D)؟**

هذه مسألة تحسين بقيد (Constrained Optimization):

$$\min_{N, D} L(N, D) \quad \text{subject to} \quad 6ND = C$$

بحل هذه المسألة (باستخدام مضاعفات لاغرانج Lagrange Multipliers على المعادلة الإمبريقية أعلاه)، توصّل فريق Chinchilla إلى أن الحل الأمثل يأخذ الشكل:

$$N_{opt}(C) = G \left(\frac{C}{6}\right)^{a}, \qquad D_{opt}(C) = G^{-1} \left(\frac{C}{6}\right)^{b}$$

حيث $a \approx b \approx 0.5$ تقريبًا (أي أن N وD يجب أن ينموا بنفس المعدل التقريبي مع زيادة C)، وG ثابت يُقدَّر تجريبيًا.

**النتيجة العملية الأشهر المستخلصة من هذا التحليل** (القاعدة الذهبية لـ Chinchilla):

$$D_{opt} \approx 20 \times N_{opt}$$

أي: **لكل معامل واحد في النموذج، يجب تدريبه على حوالي 20 رمزًا**.

### 3.3 كود: تقدير الخسارة ونقطة التحسين الأمثل


In [ ]:
import numpy as np
from scipy.optimize import minimize

class ChinchillaScalingLaw:
    """
    تطبيق عملي لمعادلة Chinchilla لتقدير الخسارة المتوقعة لأي تركيبة
    (N, D)، وإيجاد التخصيص الأمثل لأي ميزانية حوسبة معطاة.

    الثوابت أدناه مأخوذة تقريبيًا من القيم المُقدَّرة في الورقة الأصلية
    (Hoffmann et al. 2022، الجدول A.3 - تقديرات عبر Approach 3).
    """
    def __init__(self):
        # ثوابت مُقدَّرة تجريبيًا من الورقة الأصلية (بوحدات مناسبة)
        self.E = 1.69       # إنتروبيا اللغة الطبيعية التقريبية (حد أدنى للخسارة)
        self.A = 406.4      # ثابت حد النموذج
        self.alpha = 0.34   # أس حد النموذج
        self.B = 410.7      # ثابت حد البيانات
        self.beta = 0.28    # أس حد البيانات

    def predict_loss(self, N: float, D: float) -> float:
        """
        التنبؤ بالخسارة النهائية (Cross-Entropy بالـ nats) لنموذج بحجم
        N معامل مُدرَّب على D رمز، وفق معادلة Chinchilla الإمبريقية.
        """
        return self.E + (self.A / N ** self.alpha) + (self.B / D ** self.beta)

    def compute_flops(self, N: float, D: float) -> float:
        """حساب ميزانية الحوسبة التقريبية باستخدام القاعدة C ≈ 6ND"""
        return 6 * N * D

    def find_optimal_allocation(self, compute_budget: float) -> dict:
        """
        إيجاد (N_opt, D_opt) التي تُدنّي الخسارة L(N,D) تحت قيد
        ميزانية حوسبة ثابتة C = 6ND، عبر تحسين عددي مباشر
        (بديل عملي عن الحل التحليلي بمضاعفات لاغرانج).
        """
        def objective(log_N):
            # نعمل في الفضاء اللوغاريتمي لتحسين استقرار التحسين العددي
            N = np.exp(log_N[0])
            D = compute_budget / (6 * N)  # اشتقاق D من قيد الميزانية مباشرة
            if D <= 0:
                return 1e10
            return self.predict_loss(N, D)

        # تخمين أولي معقول (نبدأ من افتراض D=20N كنقطة انطلاق)
        initial_N_guess = (compute_budget / (6 * 20)) ** 0.5
        result = minimize(objective, x0=[np.log(initial_N_guess)],
                           method="Nelder-Mead")

        N_opt = np.exp(result.x[0])
        D_opt = compute_budget / (6 * N_opt)
        loss_opt = self.predict_loss(N_opt, D_opt)

        return {
            "N_opt": N_opt,
            "D_opt": D_opt,
            "D_to_N_ratio": D_opt / N_opt,
            "predicted_loss": loss_opt,
            "compute_budget": compute_budget,
        }

def demonstrate_chinchilla_tradeoff():
    """
    توضيح عملي: لنفس ميزانية الحوسبة، ماذا يحدث للخسارة إذا انحرفنا
    عن التخصيص الأمثل (نموذج كبير جدًا+بيانات قليلة، أو العكس)؟
    """
    scaling_law = ChinchillaScalingLaw()
    compute_budget = 1e21  # ميزانية حوسبة ثابتة (FLOPs) مشابهة لتجربة متوسطة

    # الحل الأمثل وفق التحسين
    optimal = scaling_law.find_optimal_allocation(compute_budget)
    print("=== التخصيص الأمثل (Compute-Optimal) ===")
    print(f"N_opt (عدد المعاملات): {optimal['N_opt']:.3e}")
    print(f"D_opt (عدد الرموز): {optimal['D_opt']:.3e}")
    print(f"النسبة D/N: {optimal['D_to_N_ratio']:.2f}  (قريبة من 20 كما تتوقع Chinchilla)")
    print(f"الخسارة المتوقعة: {optimal['predicted_loss']:.4f}\n")

    # سيناريوهات بديلة بنفس ميزانية الحوسبة بالضبط لكن بتوزيع مختلف
    print("=== مقارنة سيناريوهات بديلة بنفس ميزانية الحوسبة ===")
    scenarios = {
        "نموذج ضخم جدًا + بيانات قليلة (مثل الاتجاه القديم لـ GPT-3)": 5.0,
        "التخصيص الأمثل (Chinchilla-style)": 1.0,
        "نموذج صغير + بيانات هائلة جدًا": 0.2,
    }

    for name, multiplier in scenarios.items():
        N = optimal["N_opt"] * (multiplier ** 0.5)
        D = compute_budget / (6 * N)
        loss = scaling_law.predict_loss(N, D)
        print(f"\n{name}:")
        print(f"   N = {N:.3e}, D = {D:.3e}, D/N = {D/N:.1f}")
        print(f"   الخسارة المتوقعة = {loss:.4f} "
              f"({'أفضل' if loss < optimal['predicted_loss'] else 'أسوأ'} "
              f"من الأمثل بفارق {abs(loss - optimal['predicted_loss']):.4f})")

demonstrate_chinchilla_tradeoff()


**ناتج توضيحي متوقَّع** (تقريبي، يعتمد على القيم الدقيقة للثوابت):


```text
=== التخصيص الأمثل (Compute-Optimal) ===
N_opt (عدد المعاملات): 9.5e+09
D_opt (عدد الرموز): 1.8e+11
النسبة D/N: 19.4  (قريبة من 20 كما تتوقع Chinchilla)
الخسارة المتوقعة: 2.0564

=== مقارنة سيناريوهات بديلة بنفس ميزانية الحوسبة ===

نموذج ضخم جدًا + بيانات قليلة (مثل الاتجاه القديم لـ GPT-3):
   N = 2.1e+10, D = 7.9e+10, D/N = 3.7
   الخسارة المتوقعة = 2.0729 (أسوأ من الأمثل بفارق 0.0165)

التخصيص الأمثل (Chinchilla-style):
   N = 9.5e+09, D = 1.8e+11, D/N = 19.4
   الخسارة المتوقعة = 2.0564 (أفضل من الأمثل بفارق 0.0000)

نموذج صغير + بيانات هائلة جدًا:
   N = 4.2e+09, D = 4.0e+11, D/N = 94.3
   الخسارة المتوقعة = 2.0892 (أسوأ من الأمثل بفارق 0.0328)
```


الدرس العملي واضح: **الانحراف في أي اتجاه عن النسبة المثلى (حوالي D≈20N) يُكلِّفك جودة نموذج أقل بنفس ميزانية الحوسبة بالضبط**.

### 3.4 تحذير مهم: Compute-Optimal ≠ Inference-Optimal

> ⚠️ **خطأ شائع كبير في فهم Chinchilla**: الكثيرون يُسيئون فهم نتيجة Chinchilla كقاعدة مطلقة "يجب أن تدرّب دائمًا بنسبة D=20N بالضبط". لكن الورقة توضّح أن هذا هو التخصيص الأمثل **فقط إذا كان هدفك تدنية خسارة التدريب لميزانية حوسبة تدريب ثابتة**، متجاهلاً **تكلفة الاستدلال (Inference) اللاحقة**.

في الممارسة العملية الحديثة (كما فعلت Meta مع نماذج LLaMA)، النموذج بمجرد تدريبه سيُستخدَم للاستدلال **ملايين المرات** من قِبل المستخدمين. تكلفة الاستدلال تتناسب مباشرة مع حجم النموذج N (نموذج أصغر = استدلال أرخص وأسرع لكل طلب). لذلك، اختارت Meta عمدًا تدريب LLaMA على بيانات **أكثر بكثير** من النسبة المثلى لـ Chinchilla (مثال: LLaMA-2 7B دُرِّب على 2 تريليون رمز، بينما نسبة Chinchilla المثلى لنموذج 7B تقترح ~140 مليار رمز فقط — أي ما يقارب **14 ضعف** الكمية المثلى!).

**لماذا هذا منطقي رغم أنه "غير أمثل" وفق Chinchilla؟** لأن:
1. التدريب يحدث **مرة واحدة**، لكن الاستدلال يحدث **ملايين/مليارات المرات** لاحقًا.
2. نموذج أصغر مُدرَّب على بيانات أكثر بكثير من "المثالي للتدريب" قد يصل لنفس جودة نموذج أكبر مُدرَّب بالنسبة المثلى، لكن بتكلفة استدلال أقل بكثير إلى الأبد.
3. هذا يُسمَّى أحيانًا **"over-training"** بالمعنى الإيجابي: تدريب نموذج أصغر لفترة أطول من "المثالي نظريًا" لتوفير تكاليف تشغيلية ضخمة لاحقًا.


In [ ]:
def compare_training_optimal_vs_inference_optimal():
    """
    توضيح الفارق بين استراتيجية Chinchilla الكلاسيكية (الأمثل للتدريب فقط)
    واستراتيجية "over-training" الحديثة (الأمثل مع اعتبار تكلفة الاستدلال)
    كما طبّقتها Meta فعليًا في سلسلة LLaMA.
    """
    scaling_law = ChinchillaScalingLaw()

    scenarios = [
        {"name": "Chinchilla الأمثل نظريًا لـ N=7B",
         "N": 7e9, "D": 7e9 * 20},
        {"name": "LLaMA-2 7B الفعلي (over-trained عمدًا)",
         "N": 7e9, "D": 2e12},
    ]

    print(f"{'السيناريو':<45}{'N':<10}{'D':<12}{'D/N':<8}{'Loss المتوقع':<15}")
    print("-" * 90)
    for s in scenarios:
        loss = scaling_law.predict_loss(s["N"], s["D"])
        compute = scaling_law.compute_flops(s["N"], s["D"])
        print(f"{s['name']:<45}{s['N']:.1e}  {s['D']:.1e}  "
              f"{s['D']/s['N']:<8.1f}{loss:<15.4f}")
        print(f"   → ميزانية الحوسبة المطلوبة: {compute:.2e} FLOPs")

compare_training_optimal_vs_inference_optimal()


لاحظ النتيجة المثيرة: LLaMA-2 7B يحتاج ميزانية حوسبة **تدريب** أكبر بكثير من الحل "الأمثل نظريًا لنفس N"، لكن هذا استثمار واعٍ: دفع تكلفة تدريب إضافية مرة واحدة، توفيرًا لتكلفة استدلال أرخص إلى الأبد (لأن N أصغر يبقى صغيرًا، بينما Chinchilla الكلاسيكي قد يقترح N أكبر لنفس جودة النموذج لو اعتبرنا فقط تكلفة التدريب).


## 4. مثال محلول خطوة بخطوة

**السؤال**: لديك ميزانية حوسبة تساوي $C = 1 \times 10^{20}$ FLOPs. ما هو حجم النموذج الأمثل N وكمية البيانات الأمثل D وفق قاعدة Chinchilla التقريبية (D≈20N)؟

**الحل اليدوي**:

نبدأ من معادلة القيد: $C = 6ND$

نعوّض $D = 20N$ (قاعدة Chinchilla):

$$C = 6N(20N) = 120N^2$$

نحل لـ N:

$$N^2 = \frac{C}{120} = \frac{10^{20}}{120} \approx 8.33 \times 10^{17}$$

$$N \approx \sqrt{8.33 \times 10^{17}} \approx 9.13 \times 10^{8} \approx 913 \text{ مليون معامل}$$

وبالتالي:

$$D \approx 20 \times N \approx 20 \times 9.13 \times 10^8 \approx 1.83 \times 10^{10} = 18.3 \text{ مليار رمز}$$

**التحقق**: $C = 6 \times (9.13 \times 10^8) \times (1.83 \times 10^{10}) \approx 1.0 \times 10^{20}$ ✓ (يطابق الميزانية المُعطاة).


In [ ]:
def solve_chinchilla_example_manually(compute_budget: float,
                                        d_to_n_ratio: float = 20.0) -> dict:
    """
    حل تحليلي مباشر (وليس تحسينًا عدديًا) لإيجاد N وD المثاليين
    بافتراض نسبة D/N معينة (عادة 20 وفق قاعدة Chinchilla التقريبية)
    وقيد ميزانية حوسبة C = 6ND.

    الاشتقاق: C = 6ND = 6N(ratio×N) = 6×ratio×N²
    إذن: N = sqrt(C / (6×ratio))
    """
    N_opt = (compute_budget / (6 * d_to_n_ratio)) ** 0.5
    D_opt = d_to_n_ratio * N_opt
    verification_C = 6 * N_opt * D_opt

    return {
        "N_opt": N_opt,
        "D_opt": D_opt,
        "verification_compute": verification_C,
        "matches_budget": abs(verification_C - compute_budget) / compute_budget < 0.01,
    }

result = solve_chinchilla_example_manually(compute_budget=1e20)
print(f"N الأمثل: {result['N_opt']:.3e} معامل ({result['N_opt']/1e6:.0f} مليون)")
print(f"D الأمثل: {result['D_opt']:.3e} رمز ({result['D_opt']/1e9:.1f} مليار)")
print(f"التحقق من ميزانية الحوسبة: {result['verification_compute']:.3e} "
      f"(مطابق: {result['matches_budget']})")


**الناتج**:


```text
N الأمثل: 9.129e+08 معامل (913 مليون)
D الأمثل: 1.826e+10 رمز (18.3 مليار)
التحقق من ميزانية الحوسبة: 1.000e+20 (مطابق: True)
```


## 5. نصائح وحيل

- استخدم قاعدة **D≈20N** فقط كنقطة انطلاق تقريبية سريعة؛ للحصول على تقديرات دقيقة لمشروعك الخاص، يُفضَّل إجراء "تجارب نموذج مُصغَّر" (Scaling Law Experiments) بتدريب عدة نماذج صغيرة بأحجام مختلفة على نفس بيانات مشروعك، ثم ملاءمة المعادلة الإمبريقية $L(N,D)$ على بياناتك الفعلية (وليس الاعتماد فقط على ثوابت ورقة Chinchilla الأصلية التي قيست على بيانات DeepMind الخاصة).
- إذا كان الهدف النهائي **نشر النموذج على نطاق واسع** (تكلفة استدلال متكررة)، فكّر بجدية في "over-training" نموذج أصغر بنسبة D/N أعلى بكثير من 20، كما فعلت استراتيجية LLaMA.
- راقب "Loss Curve" أثناء التدريب الفعلي: إذا توقفت الخسارة عن التحسن رغم استمرار التدريب (Plateau مبكر)، فهذا مؤشر إما على عدم كفاية تنوع البيانات أو الحاجة لتعديل معدل التعلم، وليس بالضرورة "استنفاد قدرة النموذج".


## 6. أخطاء شائعة ومصائد

| الخطأ | الشرح |
|---|---|
| اعتبار D=20N قانونًا فيزيائيًا ثابتًا دائمًا | هو تقريب تجريبي لظروف ورقة Chinchilla المحددة؛ يتغير قليلاً حسب جودة البيانات والمعمارية |
| تجاهل تكلفة الاستدلال عند تصميم حجم النموذج | يؤدي لبناء نموذج كبير مكلف التشغيل بينما نموذج أصغر "over-trained" كان سيُعطي نفس الجودة بتكلفة تشغيل أقل |
| الخلط بين "ميزانية حوسبة التدريب" و"ميزانية حوسبة الاستدلال" كمفهوم واحد | هما قراران مستقلان بتريدات-أوف مختلفة تمامًا |
| افتراض أن زيادة N وحدها (بدون زيادة D) ستحسّن الأداء دائمًا | وفق المعادلة، تحسين أحد الحدين فقط (N أو D) دون الآخر يصل بسرعة لحد تناقص العوائد (Diminishing Returns) |


## 7. جدول مقارنة

| النموذج | N (معاملات) | D (رموز) | D/N | الاستراتيجية |
|---|---|---|---|---|
| GPT-3 (2020) | 175B | ~300B | ~1.7 | ما قبل Chinchilla — نموذج ضخم، بيانات نسبيًا قليلة (Undertrained) |
| Chinchilla (2022) | 70B | 1.4T | 20 | أول تطبيق صريح لقاعدة D≈20N |
| LLaMA-1 (2023) | 7B-65B | 1T-1.4T | ~20-140 | مزيج: بعض الأحجام قريبة من Chinchilla، أخرى over-trained |
| LLaMA-2 7B (2023) | 7B | 2T | ~286 | Over-training متعمَّد جدًا لتحسين كفاءة الاستدلال |
| LLaMA-3 8B (2024) | 8B | 15T | ~1875 | Over-training شديد جدًا — اتجاه الصناعة الحالي |

> 💡 لاحظ الاتجاه الواضح عبر الزمن: نسبة D/N ترتفع باستمرار كلما تقدّمنا زمنيًا، لأن الصناعة أدركت أن تكلفة الاستدلال طويلة الأمد تستحق استثمار حوسبة تدريب إضافية لبناء نماذج أصغر وأكفأ تشغيليًا.


## 8. ملخص أفضل الممارسات

- استخدم قانون Chinchilla كنقطة انطلاق لتقدير حجم النموذج/البيانات المناسبين لميزانيتك، لا كقاعدة جامدة.
- فرّق دائمًا بين "الأمثل لتدريب فقط" و"الأمثل شاملاً تكلفة الاستدلال طويلة الأمد".
- إن كان مشروعك سيُستخدَم على نطاق واسع، فكّر بجدية في تدريب نموذج أصغر على بيانات أكثر بكثير من 20×N.
- قبل الاستثمار الكامل في تدريب كبير، جرِّب قياس منحنيات Scaling على نماذج أصغر لتقدير الثوابت الخاصة ببياناتك ومعماريتك.
- راقب نسبة D/N لنماذج Open-Source الناجحة الحديثة (LLaMA-3، Qwen، Mistral) كمرجعية عملية محدَّثة.


## 9. أسئلة مقابلات

**سؤال 1**: اشرح الفرق الجوهري بين نتائج Kaplan et al. (2020) ونتائج Chinchilla (2022)، ولماذا اختلفت النتائج رغم أن كلتا الدراستين تدرسان نفس الظاهرة؟

> **الإجابة النموذجية**: ورقة Kaplan استنتجت أن حجم النموذج N هو العامل الأهم بكثير مقارنة بكمية البيانات D، مما دفع لاتجاه "النماذج الأكبر فالأكبر" (مثل GPT-3). لاحقًا، اكتشف فريق Chinchilla أن منهجية Kaplan في قياس العلاقة احتوت على مشاكل تجريبية (استخدام معدل تعلم غير مُحسَّن بشكل متسق عبر أحجام نماذج مختلفة، وعدم تجربة نطاق كافٍ من نسب N/D)، مما أدى لتقدير منحرف لأهمية N النسبية. بإجراء تجارب أوسع وأكثر دقة منهجيًا (تدريب أكثر من 400 نموذج بأحجام مختلفة مع ضبط معدل التعلم بعناية لكل حجم)، وجد فريق Chinchilla أن N وD لهما أهمية متقاربة تقريبًا، وأن التخصيص الأمثل يتطلب نموهما معًا بنفس المعدل تقريبًا، وليس تفضيل N فقط.

**سؤال 2**: لماذا دُرِّبت نماذج LLaMA على نسبة D/N أعلى بكثير (أحيانًا 100-1000) من نسبة Chinchilla المثلى (20)؟ هل هذا يعني أن فريق Meta "أخطأ" أو تجاهل أبحاث Chinchilla؟

> **الإجابة النموذجية**: لا، هذا قرار هندسي مدروس وليس خطأً. نتيجة Chinchilla تُحسِّن فقط خسارة التدريب مقابل **ميزانية حوسبة تدريب** ثابتة، متجاهلة تمامًا تكلفة الاستدلال اللاحق. لكن في سيناريو عملي حقيقي (نموذج سيُستخدَم من قِبل ملايين المستخدمين يوميًا)، تكلفة الاستدلال التراكمية طويلة الأمد قد تفوق بكثير تكلفة التدريب الإضافية لمرة واحدة. لذلك، تدريب نموذج أصغر (N أصغر، تكلفة استدلال أرخص لكل طلب) على كمية أكبر بكثير من البيانات (D أكبر) حتى يصل لمستوى جودة مشابه لنموذج Chinchilla-optimal أكبر، هو استثمار اقتصادي رابح على المدى الطويل، حتى لو كان "غير أمثل" بالمعنى الضيق لتدنية خسارة التدريب فقط.

**سؤال 3**: كيف يُشتق تقريب $C \approx 6ND$ لميزانية الحوسبة، وما الافتراضات الكامنة وراءه؟

> **الإجابة النموذجية**: لكل رمز تدريب، تتطلب التمريرة الأمامية (Forward Pass) تقريبًا $2N$ FLOP (كل معامل يُستخدَم في ضرب وجمع واحد تقريبًا)، والتمريرة الخلفية (Backward Pass) تتطلب تقريبًا ضعف ذلك أي $4N$ FLOP (لحساب تدرجات كل من الأوزان والتنشيطات الوسيطة). المجموع لكل رمز واحد هو $2N + 4N = 6N$، وبضربه في إجمالي عدد رموز التدريب D نحصل على $C \approx 6ND$. هذا تقريب يُهمِل بعض العمليات الثانوية (مثل حسابات LayerNorm/Softmax التي تكلفتها أصغر بكثير من عمليات ضرب المصفوفات الكبرى)، ويفترض ضمنًا أن معظم معاملات النموذج تُستخدَم في عمليات ضرب مصفوفات كثيفة (Dense Matmuls) كما في Transformer القياسي؛ هذا التقريب يصبح أقل دقة لمعماريات متناثرة مثل MoE، حيث عدد المعاملات **النشطة** فعليًا لكل رمز أقل بكثير من إجمالي عدد المعاملات الكلي.


## 10. تمرين عملي

باستخدام كلاس `ChinchillaScalingLaw` أعلاه:
1. احسب N_opt وD_opt لثلاث ميزانيات حوسبة مختلفة: $10^{19}$، $10^{21}$، $10^{23}$ FLOPs.
2. ارسم (أو صِف نصيًا) كيف تنمو N_opt وD_opt مع زيادة C (هل العلاقة خطية؟ تربيعية؟ جذرية؟).
3. لكل ميزانية، احسب الخسارة المتوقعة إذا كان N أكبر 5 أضعاف من N_opt (مع تعديل D وفق قيد الميزانية)، وقارنها بالخسارة عند N_opt — ماذا تستنتج عن حساسية الأداء للانحراف عن النسبة المثلى؟


## 11. مراجع للتعمّق

- Hoffmann et al. (2022), *"Training Compute-Optimal Large Language Models"* (ورقة Chinchilla الأصلية، DeepMind).
- Kaplan et al. (2020), *"Scaling Laws for Neural Language Models"* (الورقة السابقة التي صحّحتها Chinchilla).
- Touvron et al. (2023), *"LLaMA: Open and Efficient Foundation Language Models"* — لفهم استراتيجية over-training العملية.
- Touvron et al. (2023), *"Llama 2: Open Foundation and Fine-Tuned Chat Models"*.
- Sardana & Frankle (2023), *"Beyond Chinchilla-Optimal: Accounting for Inference in Language Model Scaling Laws"* — امتداد رسمي لنظرية Chinchilla يُدمِج تكلفة الاستدلال صراحة في المعادلة.

---

> 🔜 **الرد القادم**: سنكمل الجزء الخامس بالفصول **21 (التوازي الموزَّع: Data/Tensor/Pipeline Parallelism، ZeRO، FSDP)**، **22 (Checkpointing ومراقبة التدريب)**، و**23 (خط إنتاج ما بعد التدريب الكامل: Base → SFT → RM → RLHF/DPO)**، يليها **المشروع الختامي للجزء الخامس بالكامل**: بناء نموذج لغوي 50-200 مليون معامل من الصفر، من تنظيف البيانات وحتى نموذج محادثة جاهز.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — بناء LLM من الصفر / الفصل 20: قوانين التوسّع: Chinchilla.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
